# Hetionet 데이터 다운로드 및 서브그래프 추출

이 노트북은 Hetionet v1.0 생물의학 지식 그래프에서 **Gene–Context–Disease 서브그래프**를
추출하는 전체 과정을 다룹니다.

**파이프라인 전체 흐름:**
```
Hetionet v1.0                    TCGA-KIRC
(47K nodes, 2.25M edges)         (RNA-seq, ~610 samples)
        │                              │
   [Part 1] 구조 탐색            [Part 3] 다운로드 + TPM 통일
        │                              │
        │                         [Part 4] Gene ID 매칭
        │                         (Ensembl → Entrez)
        │                              │
        └──────── [Part 5] ────────────┘
                    │
        Gene::Entrez 기준 서브그래프
        (Pathway / Biological Process)
```

### 이전 버전에서 달라진 점

| 항목 | 이전 | 현재 |
|------|------|------|
| Gene ID 매칭 | 별도 단계(step4)로 분리 | **이 노트북 안에서 수행** (Part 4) |
| Context 레이어 | `GpPW` + `GpBP` 동시 유지 | **`GRAPH_CONTEXT` 하나로 선택** |
| 결과 저장 | `data/subgraph_*.tsv` 한 벌 | **`results/<context>/` 로 분리 저장** |
| 두 방식 비교 | 없음 | **자동 실행 + 비교표 생성** |
| 기록 | 없음 | **README.md 자동 갱신** |
| 백업 | 없음 | **Google Drive 동기화** |

구조 탐색(Part 1)·서브그래프 추출(Part 2)·KIRC 다운로드(Part 3) 코드는 원본 그대로 유지했습니다.

**이번 단계에서 다루지 않는 것:** variance scoring, candidate ranking, ML, GNN, biomarker selection.

---
## 0. 환경 설정

원본의 import 블록에 Google Drive 마운트와 프로젝트 경로 탐색을 추가했습니다.
`DATA_DIR`은 원본과 동일하게 유지되므로 기존 `data/` 폴더를 그대로 재사용합니다.

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from collections import Counter
import os, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

print(f"pandas {pd.__version__}")
print(f"networkx {nx.__version__}")
print(f"numpy {np.__version__}")
print(f"colab {IN_COLAB}")

### 0-1. Google Drive 마운트

Colab에서 실행할 때만 마운트합니다. 로컬 실행이면 건너뛰고, 그 사실을 그대로 출력합니다
(마운트하지 않았는데 성공한 것처럼 표시하지 않습니다).

In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/KIRC_Hetionet_Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("mounted ->", DRIVE_PROJECT_DIR)
else:
    print("[skip] Colab이 아니므로 Google Drive를 마운트할 수 없습니다.")
    print("       로컬에서 백업 동작을 확인하려면 KIRC_HETIONET_DRIVE_DIR 환경변수를 지정하세요.")

### 0-2. 프로젝트 경로

파이프라인 코드는 리포지토리(`src/kirc_hetionet/`)에 있습니다.
Colab에서는 `/content`로 clone하고, 로컬에서는 상위 폴더를 거슬러 올라가 찾습니다.

In [ ]:
REPO_URL = "https://github.com/kwak-lazy/solid-revice.git"
BRANCH = "claude/determined-lovelace-mj90py"


def locate_project() -> Path:
    """config.py와 src/를 가진 프로젝트 루트를 찾는다."""
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "config.py").exists() and (candidate / "src").exists():
            return candidate
    if IN_COLAB:
        target = Path("/content/solid-revice")
        if not (target / "config.py").exists():
            !git clone --branch {BRANCH} --depth 1 {REPO_URL} {target}
        return target
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. 리포지토리 안에서 실행하거나 "
        "KIRC_HETIONET_PROJECT_DIR를 지정하세요."
    )


PROJECT_DIR = locate_project()
os.environ.setdefault("KIRC_HETIONET_PROJECT_DIR", str(PROJECT_DIR))
for p in (str(PROJECT_DIR), str(PROJECT_DIR / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

import config
from kirc_hetionet import (
    hetionet, kirc, gene_mapping, context, subgraph, readme_log, report,
    drive as drive_sync,
)

config.ensure_dirs()

# 원본과 동일한 data/ 경로를 유지 (기존 파일 재사용)
DATA_DIR = str(config.RAW_DIR)

print("project dir :", config.PROJECT_DIR)
print("data dir    :", DATA_DIR)
print("results dir :", config.RESULTS_DIR)

---
## 1. Hetionet 데이터 다운로드

### Hetionet이란?

Hetionet v1.0은 29개 공개 데이터베이스를 통합한 **생물의학 이종 네트워크(heterogeneous biomedical network)**입니다.

- **47,031개 노드** (11가지 타입: Gene, Disease, Compound, Pathway 등)
- **2,250,197개 엣지** (24가지 관계 타입)

데이터는 두 파일로 배포됩니다:

| 파일 | 내용 | 크기 |
|------|------|------|
| `nodes.tsv` | 모든 노드 정보 (id, name, kind) | ~2MB |
| `edges.sif.gz` | 모든 엣지 정보 (source, metaedge, target) | ~12MB (압축) |

### 다운로드 시 주의사항

`edges.sif.gz`는 **Git LFS**로 관리됩니다. `raw.githubusercontent.com`으로 받으면 133바이트짜리 LFS 포인터만 받게 됩니다. 반드시 `media.githubusercontent.com` 경로를 사용해야 합니다.

> **변경점:** 원본의 두 다운로드 셀을 `hetionet.download_hetionet()` 한 줄로 묶었습니다.
> 내부 동작은 동일합니다 — `nodes.tsv`는 일반 raw URL, `edges.sif.gz`는 **media URL**로
> 받고, 추가로 **sha256 체크섬을 검증**합니다(133바이트 LFS 포인터를 받아도 바로 걸립니다).
> 이미 받은 파일은 건너뜁니다.

In [ ]:
paths = hetionet.download_hetionet()

---
## 2. 원본 데이터 로드 및 기본 구조 파악

### 파일 구조

**nodes.tsv** — 3개 컬럼:
- `id`: `타입::식별자` 형식 (예: `Gene::7157`)
- `name`: 사람이 읽을 수 있는 이름 (예: `TP53`)
- `kind`: 노드 타입 (예: `Gene`)

**edges.sif** — 3개 컬럼:
- `source`: 출발 노드 ID (예: `Gene::9021`)
- `metaedge`: 엣지 타입 약어 (예: `GpPW`)
- `target`: 도착 노드 ID (예: `Pathway::PC7_6941`)

In [ ]:
# 데이터 로드 (원본과 동일 / 헤더 행 혼입 제거도 그대로)
nodes = hetionet.load_nodes()
edges = hetionet.load_edges()
edges = edges[edges["metaedge"] != "metaedge"]

print(f"전체 노드 수: {len(nodes):,}")
print(f"전체 엣지 수: {len(edges):,}")

In [ ]:
# nodes.tsv 구조 확인
print("=== nodes.tsv ===")
print(f"컬럼: {list(nodes.columns)}")
print()
nodes.head(10)

In [ ]:
# edges.sif 구조 확인
print("=== edges.sif ===")
print(f"컬럼: {list(edges.columns)}")
print()
edges.head(10)

---
## 3. 노드 타입 분석 (11종)

Hetionet에는 11가지 노드 타입이 있습니다. 각 타입이 생물학적으로 무엇을 의미하는지, 그리고 어떤 **식별자 체계**를 쓰는지 파악하는 것이 중요합니다.

In [ ]:
# 노드 타입별 개수
node_counts = nodes["kind"].value_counts()

print(f"{'타입':<25s} {'개수':>8s}")
print("-" * 35)
for kind, count in node_counts.items():
    print(f"{kind:<25s} {count:>8,}")

print(f"{'─'*35}")
print(f"{'합계':<25s} {node_counts.sum():>8,}")

### 우리가 사용할 노드 타입 4종

| 타입 | 개수 | 의미 | 식별자 체계 |
|------|------|------|-------------|
| **Gene** | 20,945 | 유전자 | Entrez Gene ID (정수) |
| **Biological Process** | 11,381 | 생물학적 과정 | GO ID |
| **Pathway** | 1,822 | 생물학적 경로 | PC7_xxx / WPxxx |
| **Disease** | 137 | 질병 | DOID |

나머지 7종(Compound, Side Effect, Molecular Function, Cellular Component, Symptom, Anatomy, Pharmacologic Class)은 약물 분석이나 해부학 연구에 필요한 것들로, 현재 Gene-Pathway-Disease 분석에서는 제외합니다.

---
## 4. 식별자(ID) 체계 분석

Hetionet의 `id` 컬럼은 **이름이 아니라 표준 식별자(ID)**를 사용합니다. 이것은 매우 중요한 사실인데, 나중에 TCGA-KIRC 데이터와 매칭할 때 이 ID 체계를 기준으로 매핑해야 하기 때문입니다.

### 4-1. Gene 노드의 ID 체계

In [ ]:
# Gene 노드의 ID 형식 확인
# Gene::7157 에서 7157은 Entrez Gene ID (NCBI가 부여한 고유 정수 식별자)

genes = nodes[nodes["kind"] == "Gene"].copy()
genes["entrez_id"] = genes["id"].str.split("::").str[1].astype(int)

print(f"Gene 노드 수: {len(genes):,}")
print(f"\nID 형식: Gene::<Entrez_Gene_ID>")
print(f"\n예시:")
print(f"{'Hetionet ID':<20s} {'Entrez ID':>12s}   {'Gene Symbol'}")
print("-" * 50)
for _, row in genes.head(8).iterrows():
    print(f"{row['id']:<20s} {row['entrez_id']:>12d}   {row['name']}")

print(f"\n★ TCGA-KIRC는 Ensembl ID (예: ENSG00000141510)를 사용")
print(f"  → Entrez ↔ Ensembl 매핑이 반드시 필요 (step4에서 수행)")

### 4-2. Pathway 노드의 ID 체계

Pathway 노드에는 **두 가지 접두사 체계**가 혼재합니다:

- `PC7_xxxxx` — Pathway Commons 7의 내부 ID. Reactome, KEGG, BioCyc 등 여러 경로 DB를 통합한 것 (1,528개)
- `WPxxx_rNNNNN` — WikiPathways ID + revision 번호 (294개)

이름이 아닌 **ID 기반 매칭**이 안정적입니다 (이름은 DB마다 표기가 다를 수 있음).

In [ ]:
# Pathway 노드의 ID 형식 확인
pathways = nodes[nodes["kind"] == "Pathway"].copy()
pathways["raw_id"] = pathways["id"].str.split("::").str[1]
pathways["db_prefix"] = pathways["raw_id"].str.extract(r"^([A-Za-z]+\d*)")

print(f"Pathway 노드 수: {len(pathways):,}")

# 접두사별 개수
prefix_counts = pathways["db_prefix"].value_counts()
print(f"\n접두사별 분류:")
for prefix, count in prefix_counts.items():
    if prefix == "PC7":
        db_name = "Pathway Commons 7 (Reactome/KEGG/BioCyc 등 통합)"
    elif prefix.startswith("WP"):
        db_name = "WikiPathways"
    else:
        db_name = "Unknown"
    print(f"  {prefix}: {count:,}개 — {db_name}")

# 예시 출력
print(f"\n예시:")
for _, row in pathways.head(5).iterrows():
    print(f"  {row['id']:<35s} → {row['name']}")

### 4-3. Disease 노드의 ID 체계

In [ ]:
# Disease 노드 확인 — Disease Ontology ID (DOID) 사용
diseases = nodes[nodes["kind"] == "Disease"].copy()

print(f"Disease 노드 수: {len(diseases):,}")
print(f"\n신장(kidney) 관련 Disease:")

kidney_mask = diseases["name"].str.lower().str.contains("kidney|renal|clear cell")
kidney_diseases = diseases[kidney_mask]

for _, row in kidney_diseases.iterrows():
    print(f"  {row['id']:<25s} → {row['name']}")

print(f"\n★ TCGA-KIRC(Kidney Renal Clear Cell Carcinoma)와 직접 관련된 노드:")
print(f"  Disease::DOID:263 (kidney cancer)")

---
## 5. 엣지 타입(metaedge) 분석 (24종)

### 메타엣지 약어 읽는 법

Hetionet의 엣지 약어는 규칙적입니다:
- **앞글자** = source 노드 타입의 머리글자
- **뒷글자** = target 노드 타입의 머리글자
- **중간** = 관계를 나타내는 약어
- `>` = 방향성 표시

예시:
- `GpPW` = **G**ene **p**articipates **P**ath**W**ay
- `Gr>G` = **G**ene **r**egulates **G**ene (방향 있음)
- `DaG`  = **D**isease **a**ssociates **G**ene

> **변경점:** 원본의 `METAEDGE_NAMES` 딕셔너리(24종)를 `config.METAEDGE_NAMES`로 옮겼습니다.
> 노트북·스크립트가 같은 표를 쓰도록 하기 위한 것이고 내용은 동일합니다.

In [ ]:
METAEDGE_NAMES = config.METAEDGE_NAMES   # 원본과 동일한 24종 표

edge_counts = edges["metaedge"].value_counts()

print(f"{'약어':<6s} {'개수':>10s}   {'의미'}")
print("─" * 65)
for me, count in edge_counts.items():
    print(f"{me:<6s} {count:>10,}   {METAEDGE_NAMES.get(me, '???')}")

print(f"{'─'*65}")
print(f"{'합계':<6s} {edge_counts.sum():>10,}")

---
## 6. TCGA-KIRC 발현 데이터 다운로드

원본 Part 3입니다. **순서만 앞으로 당겼습니다** — Gene ID 표준화(Part 7)가 KIRC의
Ensembl ID 목록을 필요로 하기 때문입니다.

| 파일 | 내용 | 크기 |
|------|------|------|
| `TCGA-KIRC.star_tpm.tsv.gz` | STAR 정렬 후 TPM, `log2(TPM + 1)` 변환 | ~178 MB |
| `TCGA-KIRC.clinical.tsv.gz` | 임상 정보 | ~150 KB |

### 식별자 체계

발현 행렬의 행 인덱스는 **버전 접미사가 붙은 Ensembl ID**입니다 (예: `ENSG00000141510.17`).
Hetionet Gene 노드는 Entrez ID를 쓰므로 두 체계를 잇는 매핑이 필요합니다.
원본에서는 이 작업을 별도 단계(step4)로 미뤘지만, **이제 Part 7에서 바로 수행합니다.**

### 다운로드 시 주의사항

Xena 공식 호스트(`gdc.xenahubs.net`)가 막힌 망에서는 동일 데이터를 서빙하는
S3 버킷(`gdc-hub.s3.us-east-1.amazonaws.com`)으로 폴백합니다.
원본의 `fetch_xena()`를 그대로 `kirc.fetch_xena()`로 옮겼습니다.

In [ ]:
t0 = time.time()
kirc_paths = kirc.download_kirc()
print(f"\n소요시간: {time.time()-t0:.1f}초")

In [ ]:
# ── 발현 행렬 로드 및 구조 확인 (원본과 동일) ──
kirc_expr, kirc_expr_path = kirc.load_kirc_expression()

print(f"발현 행렬: {kirc_expr.shape[0]:,} genes x {kirc_expr.shape[1]:,} samples")
print(f"값 범위: {kirc_expr.to_numpy().min():.2f} ~ {kirc_expr.to_numpy().max():.2f}  (log2(TPM+1))")
print(f"\n행 인덱스 = 버전 접미사가 붙은 Ensembl ID:")
print(f"  {list(kirc_expr.index[:3])}")
print(f"\n열 = TCGA 바코드:")
print(f"  {list(kirc_expr.columns[:3])}")

kirc_expr.iloc[:5, :4]

In [ ]:
# ── 시료 종류 분포 (원본과 동일: 바코드 4번째 필드) ──
print("[시료 종류]")
print(kirc.sample_type_counts(kirc_expr).to_string(index=False))

kirc_clin = kirc.load_kirc_clinical()

---
## 7. Gene ID 표준화 — Ensembl → Entrez → `Gene::Entrez`

원본에서 "step4에서 별도로 진행"이라고 남겨두었던 부분입니다.

```
KIRC Ensembl Gene ID  (ENSG00000141510.17)
        │  ① 버전 접미사 제거
        ▼
ENSG00000141510
        │  ② HGNC complete set (approved) 조인
        ▼
Entrez Gene ID  (7157)  + gene_symbol (TP53)
        │  ③ "Gene::" + entrez
        ▼
Hetionet Gene::7157
```

### 두 가지 원칙

1. **최종 join key는 `Gene::Entrez`**입니다. `gene_symbol`은 annotation과 사람이 결과를
   읽기 위한 용도로만 쓰고, 조인에는 절대 쓰지 않습니다.
   (symbol은 DB·버전마다 표기가 달라 조인 키로 불안정합니다.)
2. **기존 매핑 결과가 있으면 재생성하지 않습니다.** `kirc_gene_mapping_all.tsv`,
   `kirc_hetionet_gene_nodes.tsv`, `kirc_expression_standardized.tsv.gz`를 먼저 찾고,
   쓸 수 있으면 그대로 재사용합니다. 기존 파일은 **덮어쓰지 않습니다.**

### mapping_status

| 값 | 의미 |
|---|---|
| `mapped` | Hetionet Gene 노드까지 도달 |
| `no_hgnc` | HGNC complete set에 없는 Ensembl ID |
| `no_entrez` | HGNC 레코드에 Entrez ID가 없음 |
| `not_in_hetionet` | Entrez는 있으나 Hetionet Gene 노드가 없음 |

In [ ]:
# 기존 산출물이 있으면 먼저 재사용한다.
artifacts = gene_mapping.find_existing_mapping_artifacts()

In [ ]:
kirc_ensembl_ids = kirc.kirc_ensembl_ids(kirc_expr)
print(f"KIRC Ensembl IDs (버전 제거, 중복 제거): {len(kirc_ensembl_ids):,}")

std = gene_mapping.standardize_gene_ids(kirc_ensembl_ids)
mapping = std["mapping"]
validation = std["validation"]

# 이후 모든 그래프 조인에 쓰이는 키
kirc_gene_ids = sorted(set(std["usable"]["hetionet_gene_id"].dropna()))
print(f"\n최종 graph join key (Gene::Entrez): {len(kirc_gene_ids):,}개")
mapping.head()

---
## 8. 매핑 검증

출력 항목: `KIRC genes / HGNC mapped / Entrez mapped / Hetionet mapped / Unmapped /
Duplicated / Final usable genes`

추가 점검:

- 중복 Ensembl ID
- 하나의 Ensembl ID → 여러 Entrez ID
- 하나의 Entrez ID → 여러 Ensembl ID
- Hetionet에 존재하지 않는 Entrez ID
- 최종 KIRC expression ∩ Hetionet Gene

**발견된 문제는 임의로 고치지 않습니다.** 그대로 출력하고 README 8번 항목에 기록합니다.

In [ ]:
issues = []
for issue in validation["issues"]:
    issues.append({
        "what": issue,
        "where": "gene_mapping.validate_gene_mapping()",
        "cause": "Ensembl/Entrez 식별자 체계가 1:1이 아님",
        "status": "기록만 함 (자동 수정하지 않음)",
    })

print(f"기록된 이슈: {len(issues)}건")

In [ ]:
# mapping_status별 예시 확인 — 무엇이 왜 떨어져 나갔는지 눈으로 본다.
for status in ["mapped", "not_in_hetionet", "no_entrez", "no_hgnc"]:
    part = mapping[mapping["mapping_status"] == status]
    print(f"\n[{status}] {len(part):,}건")
    if len(part):
        print(part.head(3).to_string(index=False))

---
## 9. 서브그래프 추출 — 설계

원본은 노드 4종 / 엣지 7종을 **한꺼번에** 유지했습니다:

```python
KEEP_NODE_TYPES = {"Gene", "Pathway", "Disease", "Biological Process"}
KEEP_EDGE_TYPES = {"GpPW", "GiG", "Gr>G", "DaG", "DuG", "DdG", "GpBP"}
```

여기서 **Gene→Context 레이어만** 설정값으로 분리합니다.
`GpPW`와 `GpBP`를 동시에 넣으면 두 방식을 따로 비교할 수 없기 때문입니다.

```python
GRAPH_CONTEXT = "pathway"          # 또는 "biological_process"

CONTEXT_CONFIG = {
    "pathway":            {"node_kind": "Pathway",            "metaedge": "GpPW"},
    "biological_process": {"node_kind": "Biological Process", "metaedge": "GpBP"},
}
```

나머지 5종 엣지(`GiG`, `Gr>G`, `DaG`, `DuG`, `DdG`)와 `Gene`·`Disease` 노드는
**원본 그대로** 두 context 모두에서 유지됩니다.

| | 노드 (3종) | 엣지 (6종) |
|---|---|---|
| `pathway` | Gene, Disease, **Pathway** | GiG, Gr>G, DaG, DuG, DdG, **GpPW** |
| `biological_process` | Gene, Disease, **Biological Process** | GiG, Gr>G, DaG, DuG, DdG, **GpBP** |

### 공통 함수

Pathway와 Biological Process를 각각 별도 pipeline으로 구현하지 않습니다.

```python
def get_context_nodes(nodes, graph_context):
    config = CONTEXT_CONFIG[graph_context]
    return nodes[nodes["kind"] == config["node_kind"]].copy()


def get_context_edges(edges, graph_context):
    config = CONTEXT_CONFIG[graph_context]
    return edges[edges["metaedge"] == config["metaedge"]].copy()
```

### 구현 전략: pandas 선행 필터링 (원본 유지)

2.25M 엣지를 전부 NetworkX에 올린 후 필터하면 느립니다. **pandas 벡터 연산으로 먼저
필터링**한 뒤 NetworkX에 넣습니다.

```
[느린 방법]  전체 엣지 → NetworkX 로드 → 필터링  (2분+)
[빠른 방법]  pandas 필터링 → 필요한 것만 NetworkX  (~10초)
```

In [ ]:
from config import GRAPH_CONTEXT, CONTEXT_CONFIG
from kirc_hetionet.context import (
    get_context_nodes,
    get_context_edges,
    run_context_experiment,
    run_all_context_experiments,
)

print("GRAPH_CONTEXT =", repr(GRAPH_CONTEXT))
print()
for name, cfg in CONTEXT_CONFIG.items():
    print(f"  {name:<20s} node_kind={cfg['node_kind']!r:<24s} metaedge={cfg['metaedge']!r}")
    print(f"  {'':<20s} 유지 노드: {sorted(config.keep_node_kinds(name))}")
    print(f"  {'':<20s} 유지 엣지: {sorted(config.keep_edge_types(name))}")
    print()

# 두 접근자를 현재 선택된 context에 적용
print("context nodes:", get_context_nodes(nodes, GRAPH_CONTEXT).shape)
print("context edges:", get_context_edges(edges, GRAPH_CONTEXT).shape)

---
## 10. 단일 context 실행

`run_context_experiment()` 내부가 원본의 6-1(pandas 필터) → 6-2(MultiDiGraph 구축)
→ 6-3(고립 노드 제거) 순서를 그대로 수행합니다.

`nx.MultiDiGraph`를 쓰는 이유도 원본과 같습니다:

- **Multi**: 같은 노드 쌍 사이에 여러 종류의 엣지가 존재 가능 (Gene A → Gene B에 `GiG`와 `Gr>G` 동시)
- **Di**: 방향이 있는 그래프 (`DaG`: Disease → Gene 방향)

**추가된 점:** Gene 노드를 Part 7에서 얻은 `Gene::Entrez` 집합으로 제한합니다.
KIRC에 없는 Gene은 발현값이 없어 이후 분석에 쓸 수 없기 때문입니다.

In [ ]:
single = run_context_experiment(
    GRAPH_CONTEXT,
    kirc_gene_ids=kirc_gene_ids,
    nodes=nodes,
    edges=edges,
)

G = single["graph"]          # 원본의 G와 동일한 MultiDiGraph
single["gene_context_edges"].head()

---
## 11. 서브그래프 구성 분석

원본 7~9장과 동일한 분석을, context에 무관하게 동작하도록 일반화했습니다.

In [ ]:
# ── 노드 타입별 / 엣지 타입별 개수 (원본 7장) ──
print("[노드 타입별 개수]")
print(subgraph.node_kind_counts(G).to_string(index=False))
print()
print("[엣지 타입별 개수]")
print(subgraph.edge_type_counts(G).to_string(index=False))

### 11-1. Gene–Context 연결 통계

원본 8장(`Gene-Pathway 연결 통계`)을 일반화했습니다.
Context metaedge는 Gene → Context 방향이므로, context 노드의 `in_edges` 중
해당 metaedge의 source가 Gene입니다.

In [ ]:
ctx_counts = subgraph.context_gene_counts(G, GRAPH_CONTEXT)
subgraph.describe_context_sizes(ctx_counts, GRAPH_CONTEXT, top=15)

### 11-2. Disease–Gene 연결 분석 (KIRC 관련)

원본 9장 그대로입니다. TCGA-KIRC(신장 투명세포암)와 관련된 Disease 노드가
서브그래프에 포함되어 있는지 확인합니다.

In [ ]:
kidney = subgraph.kidney_disease_report(G)
print(f"신장 관련 Disease 노드: {len(kidney)}개")
print()
print(kidney.to_string(index=False))
print()
print(f"★ TCGA-KIRC와 직접 대응되는 노드: {config.KIRC_DISEASE_ID} (kidney cancer)")

---
## 12. 두 방식 자동 실행 및 비교

`run_all_context_experiments()` 하나만 호출하면 Pathway와 Biological Process가
**같은 함수로** 차례로 실행되고 비교표까지 생성됩니다.

```python
def run_all_context_experiments():
    results = {}
    for context in ["pathway", "biological_process"]:
        results[context] = run_context_experiment(context)
    return results
```

결과는 context별 폴더에 저장되므로 서로 덮어쓰지 않습니다.

```text
results/
├── pathway/
│   ├── context_nodes.tsv
│   ├── gene_context_edges.tsv
│   ├── subgraph_nodes.tsv
│   └── subgraph_edges.tsv
├── biological_process/
│   └── (동일 4개 파일)
└── comparison/
    ├── context_comparison.tsv
    └── context_comparison_detail.tsv
```

In [ ]:
results = run_all_context_experiments(
    kirc_gene_ids=kirc_gene_ids,
    nodes=nodes,
    edges=edges,
)

In [ ]:
comparison = results["comparison"]
print(context.format_comparison(comparison))
comparison

In [ ]:
# 세부 지표 (노드 타입별 내역 포함)
pd.read_csv(config.COMPARISON_DIR / "context_comparison_detail.tsv", sep="\t")

---
## 13. 결과 저장 확인

원본 11장에 해당합니다. 저장은 `run_context_experiment()`가 이미 수행했으므로
여기서는 실제로 파일이 생겼는지 확인만 합니다.

In [ ]:
for ctx in config.ALL_GRAPH_CONTEXTS:
    d = config.results_dir(ctx)
    print(f"[{ctx}]")
    for f in sorted(d.iterdir()):
        size = f.stat().st_size
        unit = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
        print(f"  {f.name:<28s} {unit:>10s}")
    print()

print("[comparison]")
for f in sorted(config.COMPARISON_DIR.iterdir()):
    print(f"  {f.name:<28s} {f.stat().st_size/1e3:>7.1f} KB")

---
## 14. README 업데이트

`README.md`는 단순 설명 파일이 아니라 **실험 진행 기록**입니다.
실행할 때마다 Current Status / Gene ID / Graph Context / Experiment Progress /
Results / Problems / Decisions / Next Steps / Change Log가 갱신됩니다.

날짜별 블록은 **누적**되며, 같은 날짜로 다시 실행하면 그 날짜 블록만 갱신됩니다.
**실제 실행된 값만 기록합니다** — 실행하지 않은 값은 `n/a`로 남습니다.

In [ ]:
report.update_readme(
    results,
    comparison,
    std,
    {"source": "kirc_expression_matrix", "path": kirc_expr_path},
    "ensembl -> hgnc -> entrez -> hetionet",
    issues,
)
print("README updated ->", config.README_PATH)
print()
print(readme_log.get_section("Current Status"))

In [ ]:
# 기록된 결과 표 확인
print(readme_log.get_section("7. Results")[:700])

---
## 15. Google Drive 백업

`save_project_to_drive()`가 코드·노트북·README·`data/processed`·`results`를
`DRIVE_PROJECT_DIR`로 복사하고, `verify_drive_backup()`이 **디스크에서 다시 확인**합니다.

```text
Google Drive/
└── KIRC_Hetionet_Project/
    ├── notebooks/Download_and_subgraph.ipynb
    ├── README.md
    ├── config.py / src/ / scripts/
    ├── data/processed/
    └── results/{pathway,biological_process,comparison}/
```

### 실행 중인 노트북 저장에 대하여

Colab에서 실행 중인 `.ipynb`는 파일이 아니라 프론트엔드 상태입니다.
따라서 `get_ipynb`로 **현재 노트북 JSON을 받아서** 저장합니다.
그것이 불가능한 환경(일반 Jupyter, headless 실행)에서는 리포지토리의 노트북 파일을
복사하고, **어느 방법을 썼는지 그대로 출력합니다.**

저장에 실패한 파일은 `[FAIL]`로 표시되며, 성공한 것처럼 표시하지 않습니다.

In [ ]:
report_ = drive_sync.save_project_to_drive()

In [ ]:
status = drive_sync.verify_drive_backup()
print()
print("all verified:", status["all_ok"])

---
## 16. 최종 요약

### 데이터 흐름

```
Hetionet v1.0 원본                        TCGA-KIRC (star_tpm)
  47,031 노드 (11종)                        60,660 genes x 610 samples
  2,250,197 엣지 (24종)                     Ensembl ID (버전 접미사 포함)
        │                                         │
        │                                    버전 제거 → HGNC → Entrez
        │                                         │
        └───────────── Gene::Entrez ──────────────┘
                            │
             [노드 필터] Gene, Disease, <Context>
             [엣지 필터] GiG, Gr>G, DaG, DuG, DdG, <Context metaedge>
             [고립 노드 제거]
                            │
              ┌─────────────┴─────────────┐
              ▼                           ▼
        GRAPH_CONTEXT              GRAPH_CONTEXT
        = "pathway"                = "biological_process"
        (GpPW)                     (GpBP)
              │                           │
              └──────── 비교표 ───────────┘
```

### 출력 파일

| 파일 | 내용 | 다음 단계에서의 용도 |
|------|------|---------------------|
| `data/processed/kirc_gene_mapping_all.tsv` | 전체 Gene ID 매핑 + status | 매핑 근거 추적 |
| `data/processed/kirc_hetionet_gene_nodes.tsv` | 사용 가능한 Gene 노드 | 그래프 조인 키 |
| `results/<context>/context_nodes.tsv` | context 노드 + 유전자 수 | context 단위 스코어링 |
| `results/<context>/gene_context_edges.tsv` | Gene–Context 엣지 | scoring 입력 |
| `results/<context>/subgraph_nodes.tsv` | 서브그래프 노드 | 최종 그래프 구축 |
| `results/<context>/subgraph_edges.tsv` | 서브그래프 엣지 | 최종 그래프 구축 |
| `results/comparison/context_comparison.tsv` | 두 방식 비교 | context 선택 근거 |

### 핵심 사항

1. **Gene 노드는 Entrez Gene ID** 사용, TCGA-KIRC는 Ensembl ID → HGNC를 거쳐 매핑.
   최종 join key는 `Gene::Entrez`이며 gene symbol은 조인에 쓰지 않습니다.
2. **Pathway 노드는 PC7(1,528개) + WP(294개)** 두 체계 혼재 → ID 기반 매칭이 안정적.
3. **kidney cancer (DOID:263)**가 서브그래프에 포함됩니다.
4. Pathway와 Biological Process는 **하나의 pipeline**을 `GRAPH_CONTEXT`로 전환해 실행합니다.
5. pandas 선행 필터링 전략은 원본 그대로 유지했습니다.

### 다음 단계 (이번 단계에서는 구현하지 않음)

- Candidate selection / Variance calculation
- Pathway-based scoring / Biological Process-based scoring
- Candidate ranking → ML → GNN → biomarker selection